In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('foundry-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 14:27:21 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/24 14:27:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-31f0aa9a-39c1-4839-84fc-789ef073f83a;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 74ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

In [2]:
from datetime import date

from pyspark.sql import functions as F

from foundry.pipeline import TrialBalancePipeline
from foundry.enrichments import TransformationManager
from foundry.repository import (
    TrialBalanceRepository,
    TransformationRepository,
)
from foundry.config.settings import (
    CSV_TABLE_LOCATIONS, 
    POSTGRES_TABLE_LOCATIONS
)

from core.store import (
    CsvStore,
    PostgresStore
)

from atlas import AtlasClient
from reference import ReferenceClient


def display_df(df):
    display(df.toPandas())

In [7]:
business_dt = date(2025, 3, 31)

csv_store = CsvStore(spark, table_locations=CSV_TABLE_LOCATIONS)

transformation_store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(csv_store)

transformation_repository = TransformationRepository(transformation_store)
transformation_manager = TransformationManager(transformation_repository)

reference = ReferenceClient.from_db(
    spark = spark,
    fx_rate_table='reference.fx_rate',
    counterparty_table='reference.counterparty',
)

atlas = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)

pipeline = TrialBalancePipeline(
    business_dt = business_dt,
    atlas=atlas,
    repository=repository,
    reference=reference,
    transformation_manager=transformation_manager,
)

In [4]:
business_dt, batch_id = pipeline.run()

business_dt, batch_id

(datetime.date(2025, 3, 31), 22)

In [5]:
src_df = repository.read_source(business_dt=business_dt)

display_df(src_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,...,SRC_ACCT_CATEGORY,SRC_ACCT_TYPE,NORM_ACCT_SIGN,SRC_CLIENT_ID,SRC_CLIENT_NM,CPTY_REF_ID,SRC_MEASURE_NM,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD
0,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,375545,src_prev_day_bal_amt,CAD,3424081.950000000000,CAD
1,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,375545,src_current_day_debit,CAD,0E-12,CAD
2,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,375545,src_current_day_credit,CAD,-709.880000000000,CAD
3,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,375545,src_current_day_eod_balance,CAD,3423372.070000000000,CAD
4,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,375545,src_back_valued_adjustment,CAD,0E-12,CAD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,,,,src_current_day_debit,USD,0E-12,USD
92,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,,,,src_current_day_credit,USD,0E-12,USD
93,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,,,,src_current_day_eod_balance,USD,-27108.500000000000,USD
94,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,,,,src_back_valued_adjustment,USD,0E-12,USD


In [6]:
stg_df = repository.read_staging(business_dt, batch_id)

display_df(stg_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,DATACLASS,SRC_RECORD_ID,STAGING_ID,SRC_ENTITY_CD,...,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT,CR_DR_EVALUATOR
0,22,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-22-1,NKC,...,CAD,3424081.950000000000,CAD,PREVIOUS_DAY_BALANCE,REPORTABLE,CAD,3424081.950000000000,1.000000000000,3424081.950000000000,DEBIT
1,22,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-22-2,NKC,...,CAD,0E-12,CAD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,CAD,0E-12,1.000000000000,0E-12,DEBIT
2,22,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-22-3,NKC,...,CAD,-709.880000000000,CAD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,CAD,-709.880000000000,1.000000000000,-709.880000000000,DEBIT
3,22,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-22-4,NKC,...,CAD,3423372.070000000000,CAD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,CAD,3423372.070000000000,1.000000000000,3423372.070000000000,DEBIT
4,22,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-22-5,NKC,...,CAD,0E-12,CAD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,CAD,0E-12,1.000000000000,0E-12,DEBIT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,22,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-22-92,NSA,...,USD,0E-12,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT
92,22,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-22-93,NSA,...,USD,0E-12,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT
93,22,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-22-94,NSA,...,USD,-27108.500000000000,USD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,-27108.500000000000,1.000000000000,-27108.500000000000,CREDIT
94,22,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-22-95,NSA,...,USD,0E-12,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT


In [9]:
enr_df = repository.read_enrichment(business_dt, batch_id)

display_df(enr_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,DATACLASS,SRC_RECORD_ID,STAGING_ID,ENRICHMENT_ID,...,GL_ACCOUNT_DR,GL_ACCOUNT_DR_DESC,GL_ACCOUNT_CR,GL_ACCOUNT_CR_DESC,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,GL_PRODUCT_CD
0,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-19-6,ENR-250331-250331-19-1,...,100144,DUE FROM CLEARING BANKS,207150,BANK OVERDRAFTS,100144,100011,000000,LOCAL_GAAP,11392:001,000000
1,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-2,STG-250331-250331-19-12,ENR-250331-250331-19-2,...,300102,CAPITAL SURPLUS,300102,CAPITAL SURPLUS,300102,310000,505750,LOCAL_GAAP,11392:001,000000
2,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-3,STG-250331-250331-19-18,ENR-250331-250331-19-3,...,400008,INT INC - BANK DEPOSITS,400008,INT INC - BANK DEPOSITS,400008,400001,000000,LOCAL_GAAP,11392:001,000000
3,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-4,STG-250331-250331-19-24,ENR-250331-250331-19-4,...,301009,RETAINED EARNINGS (SOURCE SYSTEMS),301009,RETAINED EARNINGS (SOURCE SYSTEMS),301009,999999,A99999,LOCAL_GAAP,11392:001,000000
4,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-5,STG-250331-250331-19-30,ENR-250331-250331-19-5,...,100144,DUE FROM CLEARING BANKS,207150,BANK OVERDRAFTS,100144,100011,000000,LOCAL_GAAP,11392:001,000000
5,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-6,STG-250331-250331-19-36,ENR-250331-250331-19-6,...,198510,ACCR FEE REC - UNDERWRITING FEES,198510,ACCR FEE REC - UNDERWRITING FEES,198510,130020,000000,LOCAL_GAAP,11392:001,000000
6,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-7,STG-250331-250331-19-42,ENR-250331-250331-19-7,...,509893,CTRL - UNDERWRITING FEES,509893,CTRL - UNDERWRITING FEES,509893,411120,000000,LOCAL_GAAP,11392:001,000000
7,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-8,STG-250331-250331-19-48,ENR-250331-250331-19-8,...,301009,RETAINED EARNINGS (SOURCE SYSTEMS),301009,RETAINED EARNINGS (SOURCE SYSTEMS),301009,999999,A99999,LOCAL_GAAP,11392:001,000000
8,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-9,STG-250331-250331-19-54,ENR-250331-250331-19-9,...,100205,CASH IN BANK (IB),207150,BANK OVERDRAFTS,100205,100010,000000,LOCAL_GAAP,11392:001,000000
9,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-10,STG-250331-250331-19-60,ENR-250331-250331-19-10,...,299935,CTRL - FIRM INVENTORY,299935,CTRL - FIRM INVENTORY,299935,120014,000000,LOCAL_GAAP,11392:001,000000


In [10]:
rpt_df = repository.read_reporting(business_dt, batch_id)

display_df(rpt_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,DATACLASS,SRC_RECORD_ID,STAGING_ID,ENRICHMENT_ID,...,GL_ACCOUNT_DR,GL_ACCOUNT_DR_DESC,GL_ACCOUNT_CR,GL_ACCOUNT_CR_DESC,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,GL_PRODUCT_CD
0,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-19-1,,...,,,,,,,,,,
1,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-19-2,,...,,,,,,,,,,
2,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-19-3,,...,,,,,,,,,,
3,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-19-4,,...,,,,,,,,,,
4,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-19-5,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-19-92,,...,,,,,,,,,,
92,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-19-93,,...,,,,,,,,,,
93,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-19-94,,...,,,,,,,,,,
94,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-19-95,,...,,,,,,,,,,


In [11]:
pst_df = repository.read_posting(business_dt, batch_id)

display_df(pst_df)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,DATACLASS,SRC_RECORD_ID,STAGING_ID,ENRICHMENT_ID,...,CURRENT_DAY_CREDIT_BALANCE,CURRENT_DAY_EOD_BALANCE,BACK_VALUE_ADJUSTED_BALANCE,ADJUSTED_BALANCE,POSTING_PREVIOUS_DAY_BALANCE,POSTING_CURRENT_DAY_DEBIT_BALANCE,POSTING_CURRENT_DAY_CREDIT_BALANCE,POSTING_CURRENT_DAY_EOD_BALANCE,POSTING_BACK_VALUE_ADJUSTED_BALANCE,POSTING_ADJUSTED_BALANCE
0,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-19-6,ENR-250331-250331-19-1,...,-709.880000000000,3423372.070000000000,0E-12,3423372.070000000000,3424081.950000000000,0E-12,-709.880000000000,3423372.070000000000,0E-12,3423372.070000000000
1,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-2,STG-250331-250331-19-12,ENR-250331-250331-19-2,...,0E-12,-27500000.000000000000,0E-12,-27500000.000000000000,-27500000.000000000000,0E-12,0E-12,-27500000.000000000000,0E-12,-27500000.000000000000
2,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-3,STG-250331-250331-19-18,ENR-250331-250331-19-3,...,0E-12,-155492.000000000000,0E-12,-155492.000000000000,-155492.000000000000,0E-12,0E-12,-155492.000000000000,0E-12,-155492.000000000000
3,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-4,STG-250331-250331-19-24,ENR-250331-250331-19-4,...,0E-12,-985690.000000000000,0E-12,-985690.000000000000,-985690.000000000000,0E-12,0E-12,-985690.000000000000,0E-12,-985690.000000000000
4,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-5,STG-250331-250331-19-30,ENR-250331-250331-19-5,...,0E-12,1778940.020000000000,0E-12,1778940.020000000000,1778940.020000000000,0E-12,0E-12,1778940.020000000000,0E-12,1778940.020000000000
5,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-6,STG-250331-250331-19-36,ENR-250331-250331-19-6,...,0E-12,56250.000000000000,0E-12,56250.000000000000,56250.000000000000,0E-12,0E-12,56250.000000000000,0E-12,56250.000000000000
6,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-7,STG-250331-250331-19-42,ENR-250331-250331-19-7,...,0E-12,-73500.000000000000,0E-12,-73500.000000000000,-73500.000000000000,0E-12,0E-12,-73500.000000000000,0E-12,-73500.000000000000
7,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-8,STG-250331-250331-19-48,ENR-250331-250331-19-8,...,0E-12,-6779438.000000000000,0E-12,-6779438.000000000000,-6779438.000000000000,0E-12,0E-12,-6779438.000000000000,0E-12,-6779438.000000000000
8,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-9,STG-250331-250331-19-54,ENR-250331-250331-19-9,...,0E-12,170131525.470000000000,0E-12,170131525.470000000000,5131525.470000000000,165000000.000000000000,0E-12,170131525.470000000000,0E-12,170131525.470000000000
9,19,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-10,STG-250331-250331-19-60,ENR-250331-250331-19-10,...,0E-12,0.090000000000,0E-12,0.090000000000,0.090000000000,0E-12,0E-12,0.090000000000,0E-12,0.090000000000


In [10]:
spark.stop()